# 14 — Advanced Testing: Properties, Integration, Fakes, and CI

Reliable tests focus on behavioral contracts, invariants, independent fixtures, and realistic boundaries. Use stdlib unittest for executable examples; learn how property-based thinking extends beyond fixed examples.

**Level:** Advanced · **Study time:** 90–150 minutes · **Runtime:** Python 3.11+, standard library. Complete Basics and Intermediate before attempting this lesson.

## Outcomes

- Write invariants and table-driven tests
- Use deterministic random generation for test cases
- Test adapters with temporary databases and fakes
- Explain a layered testing strategy

**Study approach:** Predict, run, alter, and test. Consider state lifetime, invalid input, performance, and how to reason about this feature in a larger system.

## Guided theory and runnable examples

All cells work without external services. Time measurements, if any, are observations on your environment rather than universal performance facts.

### 1. Property-style reasoning

A property describes behavior across many valid inputs. Rather than proving correctness with a single example, generate reproducible cases and assert invariants. External libraries like Hypothesis can automate shrinking, but aren't required here.

In [ ]:
import random

def reverse_twice(values):
    return list(reversed(list(reversed(values))))

rng = random.Random(42)
for _ in range(30):
    values = [rng.randint(-100, 100) for _ in range(rng.randint(0, 20))]
    assert reverse_twice(values) == values
print("30 generated cases passed")

**Design review:** What property would catch a sorting implementation that loses duplicates?

### 2. Table-driven boundaries

Table-driven tests collect expected values and edge cases. unittest.subTest reports which case fails. Focus on empty, negative, very large, and invalid inputs—not only the happy path.

In [ ]:
import unittest

def clamp(number, low, high):
    if low > high:
        raise ValueError("invalid range")
    return min(max(number, low), high)

class ClampTests(unittest.TestCase):
    def test_cases(self):
        for value, expected in [(-1, 0), (4, 4), (20, 10)]:
            with self.subTest(value=value):
                self.assertEqual(clamp(value, 0, 10), expected)

result = unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ClampTests))
assert result.wasSuccessful()

**Design review:** What changes when a test uses random inputs without recording the seed?

### 3. Integration tests and temporary storage

Integration tests exercise multiple real components together, e.g. a repository and SQLite connection. In-memory databases avoid persistent side effects. Tests should still include transaction and constraint behavior.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE items(id INTEGER PRIMARY KEY, code TEXT UNIQUE)")
    db.execute("INSERT INTO items(code) VALUES (?)", ("A",))
    try:
        db.execute("INSERT INTO items(code) VALUES (?)", ("A",))
    except sqlite3.IntegrityError:
        print("Duplicate rejected")
    db.rollback()
    assert db.execute("SELECT COUNT(*) FROM items").fetchone()[0] == 0
finally:
    db.close()

**Design review:** Why did rollback remove the first insert too, and how would an earlier commit change that?

### 4. Mocks versus behavior

Mocks are appropriate at external boundaries (network, email, time). Avoid over-mocking internal helper functions, which makes tests fragile during refactoring. Check outcomes and meaningful interactions.

In [ ]:
from unittest.mock import Mock

def notify(service, message):
    return service.send(message)

fake = Mock()
fake.send.return_value = "queued"
assert notify(fake, "ready") == "queued"
fake.send.assert_called_once_with("ready")
print("Fake transport verified")

**Design review:** Why is a fake useful for simulating network failure deterministically?

## Independent challenges

For each challenge, describe an invariant, implement it, and verify at least one edge case. Avoid external dependencies unless a task explicitly says otherwise.

**Challenge 1.** Generate 100 fixed-seed inputs to test a deduplication invariant.

In [ ]:
# Challenge 1
# TODO: Write implementation and assertions.

**Challenge 2.** Write a transaction integration test showing rollback after a constraint violation.

In [ ]:
# Challenge 2
# TODO: Write implementation and assertions.

**Challenge 3.** Create a fake clock and fake network transport for a time-sensitive workflow.

In [ ]:
# Challenge 3
# TODO: Write implementation and assertions.

**Challenge 4.** Design CI stages for formatting, type checking, unit, integration, and notebook tests.

In [ ]:
# Challenge 4
# TODO: Write implementation and assertions.

## Failure modes and checkpoint

- A passing suite cannot establish correctness for every possible input.
- Randomized tests must be reproducible and report seeds.
- Mock external boundaries, not every internal implementation detail.

Can you explain why the chosen design is safe? What are its scaling limits? Is there a simpler solution? Restart the kernel and run all cells before proceeding.